In [ ]:
%%capture
!pip install --upgrade unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024   # plenty for short Q&A pairs
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",  # Unsloth's linearized version
    max_seq_length = max_seq_length,
    load_in_4bit = True,    # QLoRA — this is what makes T4 possible
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Gpt_Oss patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


model.safetensors.index.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/3387 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/22.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/15.1k [00:00<?, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                    # adapter rank — 8 is fine for narrow tasks
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # 30% less VRAM
    random_state = 3407,
)

Unsloth: Detected MoE model with num_experts = 32 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files = "asset_qa_v4_final.jsonl", split = "train")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize = False, add_generation_prompt = False
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched = True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/303 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,   # effective batch size 4
        warmup_steps = 5,
        num_train_epochs = 3,              # 2–3 epochs for fact recall
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/303 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 199998, 'pad_token_id': 200017}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 303 | Num Epochs = 3 | Total steps = 228
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 7,962,624 of 20,922,719,808 (0.04% trained)


Step,Training Loss
1,7.446339
2,6.730406
3,6.211797
4,5.302318
5,4.549861
6,3.979734
7,4.054181
8,3.427503
9,3.062112
10,2.477268


Step,Training Loss
1,7.446339
2,6.730406
3,6.211797
4,5.302318
5,4.549861
6,3.979734
7,4.054181
8,3.427503
9,3.062112
10,2.477268


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-228/tokenizer_config.json.


In [ ]:
from transformers import TextStreamer

test_questions = [
  "what is this server IP 10.10.8.16",
]

for q in test_questions:
    print(f"\n{'='*60}\nQ: {q}")
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": q}],
        add_generation_prompt = True,
        return_tensors = "pt", return_dict = True,
        reasoning_effort = "low",
    ).to(model.device)
    _ = model.generate(**inputs, max_new_tokens = 150,
                       do_sample = False,
                       streamer = TextStreamer(tokenizer, skip_prompt = True))


Q: what is this server IP 10.10.8.16


NameError: name 'tokenizer' is not defined